# DFA Minimization

This notebook introduces the idea of minimizing a deterministic finite automaton (DFA). A DFA may contain states that behave in exactly the same way for every possible future input. Such states are redundant, and we can merge them without changing the language recognized by the automaton.

In this notebook, we will study the theory behind equivalent states, learn the partition-refinement method, and implement a simple minimization algorithm in Python.


## 👨‍💻 Author

### **Muhammad Ali**

[![GitHub](https://img.shields.io/badge/GitHub-AliMuhammad78-181717?style=flat-square\&logo=github\&logoColor=white)](https://github.com/AliMuhammad78)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-Muhammad%20Ali-0A66C2?style=flat-square\&logo=linkedin\&logoColor=white)](https://www.linkedin.com/in/muhammad-ali-91294a290)
[![Kaggle](https://img.shields.io/badge/Kaggle-ali98muhammad45-20BEFF?style=flat-square\&logo=kaggle\&logoColor=white)](https://www.kaggle.com/ali98muhammad45)

## Learning objectives

By the end of this notebook, you should be able to:

- explain what it means for two DFA states to be equivalent,
- describe why some states in a DFA are redundant,
- identify accepting and nonaccepting partitions during minimization,
- apply the partition-refinement algorithm on small automata,
- implement DFA minimization in Python and test it on examples.


In [1]:
# Basic theory

# A DFA reads symbols and decides whether the string is accepted.
# If two states have exactly the same future behavior, they are equivalent.
# We can merge equivalent states without changing the language recognized.

# Example DFA representation in Python
# Each state maps an input symbol to the next state.
dfa = {
    'q0': {'0': 'q0', '1': 'q1'},
    'q1': {'0': 'q2', '1': 'q1'},
    'q2': {'0': 'q0', '1': 'q1'}
}

accepting = {'q1'}
start = 'q0'

print("States:", list(dfa.keys()))
print("Accepting states:", sorted(accepting))
print("Start state:", start)
print("From q0 on 1 goes to:", dfa['q0']['1'])

# Why minimization matters:
# - fewer states,
# - simpler machine,
# - same language.

# Equivalent states are states with the same future behavior for all input strings.
# In practice, we use a partition-refinement strategy:
# 1. Split accepting and nonaccepting states.
# 2. Repeatedly split groups based on transitions.
# 3. Stop when no further split is possible.

# This is the core idea behind DFA minimization.


States: ['q0', 'q1', 'q2']
Accepting states: ['q1']
Start state: q0
From q0 on 1 goes to: q1


## Example 1: initial partition

Suppose a DFA has states {A, B, C, D}, with accepting states {B, D} and nonaccepting states {A, C}.

The first partition is:

- Group 1: {A, C}
- Group 2: {B, D}

This split is necessary because accept vs reject states can never be equivalent.

## Example 2: refinement step

Imagine that:

- A and C both go to Group 1 on symbol 0,
- A and C both go to Group 2 on symbol 1,

Then they remain together.

But if one of them goes to Group 1 and the other to Group 2 on the same symbol, they will be split into different groups.

This is the core rule used by minimization.

## Example 3: languages with repeated patterns

A language like `(01)*` can be recognized by a DFA with repeated structure. Some states may seem different syntactically but are actually equivalent whenever they have the same future behavior.

Minimization can merge those states and produce a simpler automaton.

## Python implementation: partition refinement

The code below implements the classic minimization approach. It repeatedly splits groups until no more split is needed.

```python
# We will use a small DFA represented as:
# transitions[state][symbol] = next_state
# accepting = set of accepting states
# alphabet = list of input symbols
```

The algorithm is:

1. Start with two groups: accepting and nonaccepting states.
2. For each group, inspect transitions on each input symbol.
3. Split any group whose states go to different groups under the same symbol.
4. Repeat until no group changes.

This is the standard partition-refinement method.


In [2]:
# Partition refinement algorithm

# A DFA represented by:
# - states: all states
# - alphabet: input symbols
# - transitions: dict[state][symbol] = next_state
# - accepting: set of accepting states

states = {'q0', 'q1', 'q2', 'q3'}
alphabet = {'0', '1'}
accepting = {'q1', 'q3'}

transitions = {
    'q0': {'0': 'q0', '1': 'q1'},
    'q1': {'0': 'q2', '1': 'q1'},
    'q2': {'0': 'q0', '1': 'q3'},
    'q3': {'0': 'q2', '1': 'q1'}
}


def minimize_dfa(states, alphabet, transitions, accepting):
    """Minimize a DFA using partition refinement."""
    # Step 1: initial partition = accepting, nonaccepting
    partition = [set(accepting), set(states) - accepting]

    # Remove empty groups, if any
    partition = [group for group in partition if group]

    changed = True
    while changed:
        changed = False
        new_partition = []

        for group in partition:
            # We split this group according to how states behave on each symbol.
            groups_by_signature = {}

            for state in group:
                signature = []
                for symbol in sorted(alphabet):
                    target = transitions[state][symbol]
                    # Find which block the target belongs to
                    for i, block in enumerate(partition):
                        if target in block:
                            signature.append((symbol, i))
                            break
                signature = tuple(signature)
                groups_by_signature.setdefault(signature, set()).add(state)

            # Add all new subgroups to the new partition
            for subgroup in groups_by_signature.values():
                new_partition.append(subgroup)

            if len(groups_by_signature) > 1:
                changed = True

        partition = new_partition

    return partition


result = minimize_dfa(states, alphabet, transitions, accepting)
print("Final equivalence classes:")
for group in result:
    print(sorted(group))

print("\nInterpretation: states in the same group are equivalent and can be merged.")


Final equivalence classes:
['q1', 'q3']
['q0', 'q2']

Interpretation: states in the same group are equivalent and can be merged.


## Practice exercises

### Exercise 1: easy

A DFA has states {A, B, C}, alphabet {0, 1}, accepting states {B}, and transitions:

- A: 0 -> A, 1 -> B
- B: 0 -> C, 1 -> B
- C: 0 -> A, 1 -> B

Determine which states are equivalent and explain why.

### Exercise 2: medium

Construct a DFA with 5 states where two states are equivalent but not immediately obvious. Then minimize it by hand and check whether your answer matches the partition-refinement algorithm.

### Exercise 3: challenging

Given the following DFA:

- states = {q0, q1, q2, q3}
- alphabet = {a, b}
- accepting = {q1}
- transitions:
  - q0: a -> q1, b -> q2
  - q1: a -> q1, b -> q3
  - q2: a -> q1, b -> q2
  - q3: a -> q1, b -> q3

Find the minimized DFA and justify each merge.

### Exercise 4: implementation challenge

Modify the minimization function so it prints the partition after each refinement step. This helps you see how the algorithm gradually separates non-equivalent states.

## Final summary

The minimization algorithm works because equivalent states are those that have identical future behavior. We begin by separating accepting and nonaccepting states, then refine the partition using transition destinations. When no further split is possible, each remaining group is an equivalence class, and all states in one class can be merged.

This produces a compact DFA that recognizes the same language.


## Summary

In this notebook, we learned that:

- a DFA can have redundant states that are not necessary for language recognition,
- two states are equivalent if they have identical behavior on all possible future inputs,
- minimization separates states into groups and refines them until no more distinctions remain,
- accepting and nonaccepting states must be kept in different initial partitions,
- the minimized DFA recognizes the same language but has fewer states.

This is an important concept in automata theory because it gives a systematic way to simplify machines while preserving their behavior.
